# Figure generation

Produces all plots and sample mosaics. Each cell is self-contained — rerun individual cells when only one plot needs updating.

Outputs go to `OUTPUT_DIR`. Plots as PDF, sample mosaics as PNG.

## Configuration
Adjust the paths if needed — defaults are for the Windows setup; Linux paths are commented below each.

In [ ]:
from pathlib import Path

# Trainings-stats.json (eines pro Modell)
TRAINING_ROOT = Path(r"C:\Users\Sven\Desktop\T3_3200\3DMedDiffusion\Code\final")
# Linux: Path("/home/sven/Desktop/diffusion/Code/final")

# Per-Sample-CSVs aus evaluate.ipynb
EVAL_RESULTS = TRAINING_ROOT / "results"

# Generierte Sample-Volumina (NIfTI) aus generate.ipynb
SAMPLES_UNCOND_ROOT   = Path(r"C:\Users\Sven\Desktop\T3_3200\3DMedDiffusion\Code\unconditional\samples")
SAMPLES_PIPELINE_ROOT = Path(r"C:\Users\Sven\Desktop\T3_3200\3DMedDiffusion\Code\conditional\pipeline")

# Trainings-Checkpoints (für Trainings-Progression und Denoising-Trajektorie)
CHECKPOINT_ROOT = Path(r"C:\Users\Sven\Desktop\Data\diffusion")
# Linux: Path("/home/sven/Desktop/diffsuion/Diff 2")

# Pfad zum Source-Code (Model + Sampler)
CODE_ROOT = Path(r"C:\Users\Sven\Desktop\T3_3200\3DMedDiffusion\Code\unconditional")

# Output-Verzeichnis für die Abbildungen
OUTPUT_DIR = Path.cwd() / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# Reproduzierbare Zufalls-Sample-Auswahl
RNG_SEED = 42

# Einheitliche FID-beste Sampler pro Modell (Ergebnis aus Auswertung)
FID_BEST_SAMPLER = {
    "baseline":          "DDPM",
    "data_augmentation": "UniPC",
    "linear_schedule":   "DDPM",
    "no_attention":      "DDPM",
    "noise_prediction":  "DDPM",
    "baseline_48":       "DDPM",
    "pipeline":          "UniPC",
}

print(f"output: {OUTPUT_DIR}")

## Style and imports

In [ ]:
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "lines.linewidth": 1.2,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

MODEL_COLORS = {
    "baseline":                                   "#1f77b4",
    "data_augmentation":                          "#ff7f0e",
    "linear_schedule":                            "#d62728",
    "no_attention":                               "#2ca02c",
    "noise_prediction":                           "#9467bd",
    "baseline_48":                                "#17becf",
    "conditional":                                "#8c564b",
    "novel_conditional_segmentation_classified":  "#e377c2",
}

MODEL_LABELS = {
    "baseline":                                   r"Baseline $32^3$",
    "data_augmentation":                          "Data Augmentation",
    "linear_schedule":                            "Linearer Schedule",
    "no_attention":                               "Ohne Self-Attention",
    "noise_prediction":                           "Noise-Prediction",
    "baseline_48":                                r"Baseline $48^3$",
    "conditional":                                r"Maske $\rightarrow$ MRT",
    "novel_conditional_segmentation_classified":  r"Klasse $\rightarrow$ Maske",
}

SAMPLER_STEPS = {"DDPM": 250, "DDIM": 50, "DPMSolverPP": 25, "UniPC": 10}
SAMPLER_LABELS = {"DDPM": "DDPM (250)", "DDIM": "DDIM (50)",
                  "DPMSolverPP": "DPM-Solver++ (25)", "UniPC": "UniPC (10)"}

print("style + imports ready")

## Helpers

In [ ]:
import nibabel as nib

def load_training_stats():
    """Lädt alle stats.json unter TRAINING_ROOT/*/output_*/. Returns dict
    keyed by (size, model_name)."""
    out = {}
    for size_dir in sorted(TRAINING_ROOT.iterdir()):
        if not size_dir.is_dir() or not size_dir.name.isdigit():
            continue
        size = int(size_dir.name)
        for model_dir in sorted(size_dir.glob("output_*")):
            name = model_dir.name.replace("output_", "")
            sj = model_dir / "stats.json"
            if not sj.exists():
                continue
            with sj.open() as f:
                out[(size, name)] = json.load(f)
    return out


def load_volume(path):
    """Lädt NIfTI als float32 numpy array."""
    return nib.load(str(path)).get_fdata().astype(np.float32)


def axial_midslice(vol):
    return vol[:, :, vol.shape[2] // 2]


def norm01(img, vmin=-1.0, vmax=1.0):
    """Linear auf [0,1] für matplotlib imshow."""
    return np.clip((img - vmin) / (vmax - vmin), 0, 1)


def pick_sample_indices(n_total, n_pick, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    return sorted(rng.choice(n_total, size=n_pick, replace=False).tolist())


def list_sample_files(folder, pattern="sample_*.nii.gz"):
    return sorted(folder.glob(pattern))


print("helpers ready")

## Loss curves across all models

In [ ]:
all_stats = load_training_stats()
print(f"geladen: {len(all_stats)} Modelle")
for k in all_stats:
    print(" ", k)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7.5, 4.5))

for (size, name), st in all_stats.items():
    # 48³ baseline bekommt einen eigenen Key/Farbe
    key = "baseline_48" if (size == 48 and name == "baseline") else name
    color = MODEL_COLORS.get(key, "gray")
    label = MODEL_LABELS.get(key, name)

    pem = st["per_epoch_metrics"]
    epochs = np.arange(1, len(pem["train_losses"]) + 1)
    ax.semilogy(epochs, pem["train_losses"], "-",  color=color, label=label, alpha=0.95)
    ax.semilogy(epochs, pem["val_losses"],   "--", color=color, alpha=0.55)

ax.set_xlabel("Epoche")
ax.set_ylabel("Loss (log-Skala)")
ax.set_xlim(0, 300)
ax.grid(True, which="both", alpha=0.3, linewidth=0.5)
ax.legend(loc="upper right", ncol=2, frameon=True, framealpha=0.9)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "loss_curves.pdf")
plt.show()
print("→ loss_curves.pdf")

## Training time bar chart

In [ ]:
rows = []
for (size, name), st in all_stats.items():
    key = "baseline_48" if (size == 48 and name == "baseline") else name
    rows.append({
        "key": key,
        "label": MODEL_LABELS.get(key, name),
        "hours": st["training_info"]["total_time_hours"],
        "color": MODEL_COLORS.get(key, "gray"),
    })
df = pd.DataFrame(rows).sort_values("hours")

fig, ax = plt.subplots(1, 1, figsize=(6.5, 3.5))
bars = ax.barh(df["label"], df["hours"], color=df["color"], edgecolor="black", linewidth=0.4)
for b, h in zip(bars, df["hours"]):
    ax.text(h + 1, b.get_y() + b.get_height() / 2, f"{h:.1f}", va="center", fontsize=8)
ax.set_xlabel("Trainingsdauer [GPU-Stunden]")
ax.grid(True, axis="x", alpha=0.3, linewidth=0.5)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "training_time.pdf")
plt.show()
print("→ training_time.pdf")

## Boxplots: SSIM / PSNR / LPIPS for the 32³ models
Uses the per_sample CSVs (FID-best sampler per model).

In [ ]:
ablations = ["baseline", "data_augmentation", "linear_schedule", "no_attention", "noise_prediction"]

metric_data = {m: {} for m in ablations}
for m in ablations:
    sampler = FID_BEST_SAMPLER[m]
    csv = EVAL_RESULTS / f"per_sample_unconditional_{m}_{sampler}_32.csv"
    if not csv.exists():
        print(f"WARN: {csv.name} fehlt")
        continue
    df = pd.read_csv(csv)
    metric_data[m] = {col: df[col].values for col in ("ssim", "psnr", "lpips")}

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
metric_names = ["ssim", "psnr", "lpips"]
metric_titles = ["SSIM $\\uparrow$", "PSNR $\\uparrow$", "LPIPS $\\downarrow$"]

for ax, metric, title in zip(axes, metric_names, metric_titles):
    data = [metric_data[m][metric] for m in ablations if metric in metric_data[m]]
    labels = [MODEL_LABELS[m] for m in ablations if metric in metric_data[m]]
    bp = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.55,
                    medianprops={"color": "black"})
    for patch, m in zip(bp["boxes"], ablations):
        patch.set_facecolor(MODEL_COLORS[m])
        patch.set_alpha(0.7)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=30)
    ax.grid(True, axis="y", alpha=0.3, linewidth=0.5)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "boxplot_uncond_32.pdf")
plt.show()
print("→ boxplot_uncond_32.pdf")

## Generation performance table
Aggregates time per sample and peak VRAM from the sample stats.json files.

In [ ]:
def collect_generation_stats(image_size=32):
    """Liest stats.json neben den Sample-Ordnern (baseline-Modell, alle Sampler)."""
    rows = []
    for sampler in ("DDPM", "DDIM", "DPMSolverPP", "UniPC"):
        stats_path = SAMPLES_UNCOND_ROOT / str(image_size) / sampler / "samples_baseline" / "stats.json"
        if not stats_path.exists():
            print(f"missing: {stats_path}")
            continue
        with stats_path.open() as f:
            st = json.load(f)
        agg = st.get("aggregates", {})
        rows.append({
            "Sampler": sampler,
            "Schritte": SAMPLER_STEPS[sampler],
            "Zeit/Sample [s]": round(agg.get("mean_time_per_sample", 0), 2),
            "Gesamtzeit [min]": round(agg.get("total_time_sec", 0) / 60, 1),
            "Peak VRAM [GB]": round(agg.get("peak_gpu_memory_nvidia_smi_gb", 0), 2),
        })
    return pd.DataFrame(rows)

gen_df_32 = collect_generation_stats(32)
print("32³:")
print(gen_df_32.to_string(index=False))
gen_df_32.to_csv(OUTPUT_DIR / "generation_time_32.csv", index=False)

try:
    gen_df_48 = collect_generation_stats(48)
    print("\n48³:")
    print(gen_df_48.to_string(index=False))
    gen_df_48.to_csv(OUTPUT_DIR / "generation_time_48.csv", index=False)
except Exception as e:
    print(f"48³ skip: {e}")

print("\n→ generation_time_{32,48}.csv  (Werte für die LaTeX-Tabelle)")

## Sample grid for 32³ ablations (8 samples × 5 models)

In [ ]:
def sample_dir(size, sampler, model):
    return SAMPLES_UNCOND_ROOT / str(size) / sampler / f"samples_{model}" / "samples"

N_PICK = 8
rng_indices = pick_sample_indices(1024, N_PICK)
print(f"verwende Sample-Indizes (seed={RNG_SEED}): {rng_indices}")

In [ ]:
fig, axes = plt.subplots(len(ablations), N_PICK, figsize=(N_PICK * 1.6, len(ablations) * 1.6))

for r, m in enumerate(ablations):
    sampler = FID_BEST_SAMPLER[m]
    sdir = sample_dir(32, sampler, m)
    files = list_sample_files(sdir)
    if not files:
        print(f"WARN: {sdir} leer")
        continue
    for c, idx in enumerate(rng_indices):
        if idx >= len(files):
            continue
        vol = load_volume(files[idx])
        axes[r, c].imshow(norm01(axial_midslice(vol)), cmap="gray", vmin=0, vmax=1)
        axes[r, c].axis("off")
    axes[r, 0].set_ylabel(MODEL_LABELS[m] + f"\n({sampler})", fontsize=8, rotation=0,
                          ha="right", va="center", labelpad=40)

fig.suptitle("Generierte Samples (axialer Mittelschnitt)", fontsize=10, y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "samples_uncond_32_grid.png")
plt.show()
print("→ samples_uncond_32_grid.png")

## Resolution comparison: baseline 32³ vs 48³

In [ ]:
N_RES = 6
res_indices = pick_sample_indices(1024, N_RES, seed=RNG_SEED + 1)

fig, axes = plt.subplots(2, N_RES, figsize=(N_RES * 1.7, 3.5))

for c, idx in enumerate(res_indices):
    f32 = list_sample_files(sample_dir(32, "DDPM", "baseline"))[idx]
    f48 = list_sample_files(sample_dir(48, "DDPM", "baseline"))[idx]
    axes[0, c].imshow(norm01(axial_midslice(load_volume(f32))), cmap="gray", vmin=0, vmax=1)
    axes[1, c].imshow(norm01(axial_midslice(load_volume(f48))), cmap="gray", vmin=0, vmax=1)
    for r in (0, 1):
        axes[r, c].axis("off")

axes[0, 0].set_ylabel(r"$32^3$", fontsize=10, rotation=0, ha="right", va="center", labelpad=20)
axes[1, 0].set_ylabel(r"$48^3$", fontsize=10, rotation=0, ha="right", va="center", labelpad=20)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "samples_32_vs_48.png")
plt.show()
print("→ samples_32_vs_48.png")

## Pipeline grid (mask / MRI / overlay) for 6 samples

In [ ]:
def pipeline_dir(sampler, size=32):
    return SAMPLES_PIPELINE_ROOT / sampler / str(size) / "samples"

PIPE_SAMPLER = FID_BEST_SAMPLER["pipeline"]
N_PIPE = 6
pdir = pipeline_dir(PIPE_SAMPLER, 32)
brain_files = sorted(pdir.glob("brain_*.nii.gz"))
tumor_files = sorted(pdir.glob("tumor_*.nii.gz"))
print(f"pipeline: {len(brain_files)} brain, {len(tumor_files)} tumor in {pdir}")

if len(brain_files) >= N_PIPE and len(tumor_files) >= N_PIPE:
    pipe_indices = pick_sample_indices(len(brain_files), N_PIPE, seed=RNG_SEED + 2)
    fig, axes = plt.subplots(N_PIPE, 3, figsize=(5.5, N_PIPE * 1.8))

    for r, idx in enumerate(pipe_indices):
        brain = axial_midslice(load_volume(brain_files[idx]))
        tumor = axial_midslice(load_volume(tumor_files[idx]))

        axes[r, 0].imshow(tumor, cmap="viridis", vmin=0, vmax=tumor.max() if tumor.max() > 0 else 1)
        axes[r, 1].imshow(norm01(brain), cmap="gray", vmin=0, vmax=1)
        axes[r, 2].imshow(norm01(brain), cmap="gray", vmin=0, vmax=1)
        # Tumor-Overlay halbtransparent
        mask = np.ma.masked_where(tumor == 0, tumor)
        axes[r, 2].imshow(mask, cmap="autumn", alpha=0.5)

        for c in range(3):
            axes[r, c].axis("off")

    axes[0, 0].set_title("Maske")
    axes[0, 1].set_title("T1N (generiert)")
    axes[0, 2].set_title("Overlay")

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "samples_pipeline_grid.png")
    plt.show()
    print("→ samples_pipeline_grid.png")
else:
    print(f"skip: zu wenige pipeline files in {pdir}")

## Model-loading setup (for training progression and denoising trajectory)

In [ ]:
sys.path.insert(0, str(CODE_ROOT / "src"))

from dataclasses import dataclass
from typing import Tuple
import torch

from model.unet3d import UNet3D
from sampler.sampler import make_scheduler

@dataclass
class Config:
    data_path: str = ""
    output_path: str = ""
    image_size: int = 32
    in_channels: int = 1
    model_channels: int = 96
    channel_mult: Tuple[int, ...] = (1, 2, 4, 4)
    num_res_blocks: int = 2
    attention_resolutions: Tuple[int, ...] = (8, 4)
    num_groups: int = 8
    dropout: float = 0.0
    num_timesteps: int = 250
    beta_schedule: str = "cosine"
    prediction_type: str = "v"
    num_epochs: int = 300
    batch_size: int = 10
    learning_rate: float = 2e-4
    warmup_steps: int = 500
    ema_decay: float = 0.9999
    grad_clip: float = 1.0
    use_loss_aware_sampling: bool = True
    loss_history_size: int = 10
    snr_gamma: float = 5.0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

# Config muss in __main__ liegen, weil torch.load (weights_only=False) sie dort sucht.
sys.modules.setdefault("__main__", sys.modules[__name__])
setattr(sys.modules["__main__"], "Config", Config)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")


def load_model_from_ckpt(ckpt_path, image_size=32, attention_resolutions=(8, 4),
                         prediction_type="v", beta_schedule="cosine", device=DEVICE):
    cfg = Config(image_size=image_size, attention_resolutions=attention_resolutions,
                 prediction_type=prediction_type, beta_schedule=beta_schedule)
    model = UNet3D(cfg).to(device)
    ckpt = torch.load(str(ckpt_path), map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    if "ema" in ckpt and "shadow" in ckpt["ema"]:
        for n, p in model.named_parameters():
            if n in ckpt["ema"]["shadow"]:
                p.data.copy_(ckpt["ema"]["shadow"][n].to(device))
    model.eval()
    return model


@torch.no_grad()
def sample_with_intermediates(model, scheduler, image_size, num_steps,
                              n_capture=8, device=DEVICE, seed=42):
    """Wie sampler.sample, gibt aber n_capture Zwischen-Volumes zurück
    (inkl. Startrauschen und Endergebnis)."""
    scheduler.set_timesteps(num_steps)
    torch.manual_seed(seed)
    x = torch.randn(1, 1, image_size, image_size, image_size, device=device)

    # Schritte, an denen das Volumen gespeichert wird (Index nach jedem Step).
    capture_after = set(np.linspace(0, num_steps - 1, n_capture - 1).astype(int).tolist())

    captures = [x.clone()]  # x_T (Startrauschen)
    for i, t in enumerate(scheduler.timesteps):
        t_batch = torch.full((1,), int(t), device=device, dtype=torch.long)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            pred = model(x, t_batch).float()
        x = scheduler.step(pred, t, x).prev_sample
        if i in capture_after:
            captures.append(x.clone())
    if len(captures) < n_capture:
        captures.append(x.clamp(-1, 1).clone())
    return captures

## Training progression (baseline 32³, checkpoints across epochs)
Generates one sample per checkpoint using the same seed.

In [ ]:
CKPT_EPOCHS = [1, 10, 25, 50, 100, 150, 200, 250, 300]
PROG_SEED = 42

ckpt_dir = CHECKPOINT_ROOT / "32" / "output_baseline" / "checkpoints"

# Checkpoint-Dateinamen: Trainingscode speichert vermutlich als checkpoint_epoch_XXX.pt.
# Wenn dein Schema anders ist, hier anpassen.
def find_checkpoint(epoch):
    candidates = [
        ckpt_dir / f"checkpoint_epoch_{epoch:03d}.pt",
        ckpt_dir / f"checkpoint_epoch_{epoch}.pt",
        ckpt_dir / f"epoch_{epoch:03d}.pt",
        ckpt_dir / f"epoch_{epoch}.pt",
    ]
    if epoch == 300:
        candidates.insert(0, ckpt_dir / "final_model.pt")
    for c in candidates:
        if c.exists():
            return c
    return None

prog_samples = []
prog_labels = []
for ep in CKPT_EPOCHS:
    cp = find_checkpoint(ep)
    if cp is None:
        print(f"  skip epoch {ep}: kein Checkpoint gefunden")
        continue
    print(f"  ep {ep:3d}: lade {cp.name}")
    model = load_model_from_ckpt(cp, image_size=32, attention_resolutions=(8, 4))
    scheduler = make_scheduler("ddpm", "cosine", "v")
    scheduler.set_timesteps(250)
    torch.manual_seed(PROG_SEED)
    x = torch.randn(1, 1, 32, 32, 32, device=DEVICE)
    with torch.no_grad():
        for t in scheduler.timesteps:
            tb = torch.full((1,), int(t), device=DEVICE, dtype=torch.long)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                pred = model(x, tb).float()
            x = scheduler.step(pred, t, x).prev_sample
    prog_samples.append(x.clamp(-1, 1).squeeze().cpu().numpy())
    prog_labels.append(f"Ep. {ep}")
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

n = len(prog_samples)
fig, axes = plt.subplots(1, n, figsize=(n * 1.5, 1.8))
if n == 1:
    axes = [axes]
for ax, vol, lbl in zip(axes, prog_samples, prog_labels):
    ax.imshow(norm01(axial_midslice(vol)), cmap="gray", vmin=0, vmax=1)
    ax.set_title(lbl, fontsize=9)
    ax.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "training_progression.png")
plt.show()
print("→ training_progression.png")

## Denoising trajectory
Four samplers × {32³, 48³} baseline, identical starting seed. Each row: transition from $x_T$ (left, pure noise) to $x_0$ (right).

In [ ]:
TRAJ_SEED = 42
N_STEPS_SHOW = 8

SAMPLER_MAP = {"DDPM": ("ddpm", 250), "DDIM": ("ddim", 50),
               "DPMSolverPP": ("dpm_solver++", 25), "UniPC": ("unipc", 10)}

def gen_trajectory(image_size, attn):
    rows = {}
    cp = CHECKPOINT_ROOT / str(image_size) / "output_baseline" / "checkpoints" / "final_model.pt"
    if not cp.exists():
        cp_alt = CHECKPOINT_ROOT / str(image_size) / "output_baseline_48" / "checkpoints" / "final_model.pt"
        cp = cp_alt if cp_alt.exists() else cp
    if not cp.exists():
        print(f"  kein Checkpoint für {image_size}³ unter {cp}")
        return rows
    model = load_model_from_ckpt(cp, image_size=image_size, attention_resolutions=attn)
    for ui_name, (sampler_key, steps) in SAMPLER_MAP.items():
        scheduler = make_scheduler(sampler_key, "cosine", "v")
        n_cap = min(N_STEPS_SHOW, steps + 1)
        caps = sample_with_intermediates(model, scheduler, image_size, steps,
                                         n_capture=n_cap, seed=TRAJ_SEED)
        rows[ui_name] = [c.squeeze().cpu().numpy() for c in caps]
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return rows

print("Generiere Trajektorien 32³ …")
traj_32 = gen_trajectory(32, (8, 4))
print("Generiere Trajektorien 48³ …")
traj_48 = gen_trajectory(48, (12, 6))

In [ ]:
def plot_trajectory(traj_dict, image_size, suffix):
    if not traj_dict:
        print(f"skip plot {image_size}³ (leer)")
        return
    samplers = list(traj_dict.keys())
    n_steps = max(len(v) for v in traj_dict.values())

    fig, axes = plt.subplots(len(samplers), n_steps, figsize=(n_steps * 1.3, len(samplers) * 1.5))
    if len(samplers) == 1:
        axes = np.array([axes])

    for r, name in enumerate(samplers):
        vols = traj_dict[name]
        for c in range(n_steps):
            ax = axes[r, c]
            if c < len(vols):
                ax.imshow(norm01(axial_midslice(vols[c])), cmap="gray", vmin=0, vmax=1)
            ax.axis("off")
            if r == 0:
                ax.set_title(f"$t_{{{c}}}$", fontsize=8)
        axes[r, 0].set_ylabel(SAMPLER_LABELS[name], fontsize=8,
                              rotation=0, ha="right", va="center", labelpad=50)

    fig.suptitle(f"Denoising-Trajektorie — Baseline ${image_size}^3$ "
                 f"(links: $x_T$, rechts: $x_0$)", fontsize=10, y=1.02)
    fig.tight_layout()
    out = OUTPUT_DIR / f"denoising_trajectory_{suffix}.png"
    fig.savefig(out)
    plt.show()
    print(f"→ {out.name}")


plot_trajectory(traj_32, 32, "32")
plot_trajectory(traj_48, 48, "48")

## Sampler comparison (final results only, identical seed)
Reuses the last volumes from each trajectory for a direct side-by-side.

In [ ]:
def plot_sampler_compare(traj_dict, image_size, suffix):
    if not traj_dict:
        print(f"skip compare {image_size}³")
        return
    samplers = list(traj_dict.keys())
    fig, axes = plt.subplots(1, len(samplers), figsize=(len(samplers) * 1.8, 2))
    if len(samplers) == 1:
        axes = [axes]
    for ax, name in zip(axes, samplers):
        vol = traj_dict[name][-1]
        ax.imshow(norm01(axial_midslice(vol)), cmap="gray", vmin=0, vmax=1)
        ax.set_title(SAMPLER_LABELS[name], fontsize=9)
        ax.axis("off")
    fig.suptitle(f"Baseline ${image_size}^3$ – identisches Startrauschen", fontsize=10, y=1.05)
    fig.tight_layout()
    out = OUTPUT_DIR / f"samples_sampler_compare_{suffix}.png"
    fig.savefig(out)
    plt.show()
    print(f"→ {out.name}")


plot_sampler_compare(traj_32, 32, "32")
plot_sampler_compare(traj_48, 48, "48")

## Done
All outputs are in `OUTPUT_DIR`.

In [ ]:
print("alle Abbildungen erzeugt unter:", OUTPUT_DIR)